# Module 1 — Local LLM với Ollama

**Mục tiêu:** chạy LLM trên máy mình, hiểu vì sao Ollama là lựa chọn tốt khi cần chạy AI 100% offline (dữ liệu không rời máy).

**Yêu cầu:** đã chạy `0_setup/setup.ps1` và `0_setup/pull_models.ps1`.

---

## Vì sao Ollama (5 điểm nhấn)

1. **Cài 1 lệnh** — `winget install Ollama.Ollama`
2. **Tự tối ưu phần cứng** — detect GPU/CPU, fallback mượt
3. **Đổi model 1 dòng** — `ollama run X` → `ollama run Y`
4. **API tương thích OpenAI** — code OpenAI cũ chạy ngay
5. **Offline 100%** — sau khi pull, tắt mạng vẫn chạy

## Bước 0: Kiểm tra kết nối Ollama

In [ ]:
import ollama

# Liệt kê model đã pull về máy
models = ollama.list().models
for m in models:
    size_gb = m.size / (1024**3)
    print(f"  {m.model:30s}  {size_gb:.2f} GB")

## Bước 1: Chat đơn giản

5 dòng code → có ngay LLM trả lời. So với việc tự load `transformers`, tự quản VRAM, tự handle batching… đây là sức hấp dẫn của Ollama.

In [ ]:
MODEL = "qwen3:1.7b"  # mặc định workshop; đổi thành model bạn đã pull

response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Bạn là trợ lý hữu ích, trả lời ngắn gọn, rõ ràng."},
        {"role": "user", "content": "Giải thích RAG (Retrieval-Augmented Generation) trong 3 câu."},
    ],
    think=False,  # tắt thinking mode của Qwen3 → output sạch (không có <think>...</think>)
)

print(response.message.content)

## Bước 2: Streaming output

Hiển thị từng token khi LLM sinh — giống ChatGPT. Cải thiện UX rất nhiều với model chạy CPU.

In [ ]:
stream = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Liệt kê 5 mẹo viết email chuyên nghiệp, mỗi mẹo 1 câu."},
    ],
    stream=True,
    think=False,
)

for chunk in stream:
    print(chunk.message.content, end="", flush=True)
print()

## Bước 3: API tương thích OpenAI

**Đây là điểm bán hàng lớn nhất của Ollama:** code viết cho OpenAI ChatGPT API chạy ngay với local model — chỉ đổi `base_url`.

Điều này có nghĩa: bất kỳ thư viện nào hỗ trợ OpenAI (LangChain, LlamaIndex, Pydantic AI, OpenAI SDK chính chủ…) đều dùng được với Ollama.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",   # Ollama, KHÔNG phải openai.com
    api_key="ollama",                        # placeholder, Ollama không check key
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Bạn là trợ lý hữu ích, trả lời ngắn gọn."},
        {"role": "user", "content": (
            "Tóm tắt đoạn sau trong 1 câu: 'Cuộc họp tuần này chốt 3 việc: "
            "hoàn thiện báo cáo quý, lên kế hoạch tuyển dụng, và rà soát ngân sách.'"
        )},
    ],
    temperature=0.3,
)

print(response.choices[0].message.content)

## Bước 4: So sánh model — đổi 1 dòng

Cell sau benchmark cùng 1 prompt trên các model khác nhau. Học viên thấy ngay: 
- **Đổi model = đổi string** — không cần download lại framework, không cần restart service
- Tradeoff size vs chất lượng vs tốc độ là quyết định runtime

In [ ]:
import time

PROMPT = "Giải thích ngắn gọn cho người mới: máy học (machine learning) là gì? 3-4 câu."
MODELS_TO_TEST = ["qwen3:1.7b", "qwen3:4b", "llama3.2:3b"]

# Lấy danh sách model đã pull để skip cái chưa có
local = {m.model for m in ollama.list().models}

for model in MODELS_TO_TEST:
    if model not in local and f"{model}:latest" not in local:
        print(f"[SKIP] {model} chưa pull. Lệnh: ollama pull {model}\n")
        continue

    print(f"{'=' * 60}")
    print(f"Model: {model}")
    print("=" * 60)

    start = time.time()
    # think=False để benchmark đếm token sinh ra (không tính block <think>)
    try:
        r = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": PROMPT}],
            options={"num_predict": 200},
            think=False,
        )
    except TypeError:
        r = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": PROMPT}],
            options={"num_predict": 200},
        )
    elapsed = time.time() - start
    tok_per_sec = (r.eval_count or 0) / elapsed

    print(r.message.content)
    print(f"\n[{r.eval_count} tokens / {elapsed:.1f}s = {tok_per_sec:.1f} tok/s]\n")

## Bước 5: Modelfile — đóng gói system prompt

Modelfile giống Dockerfile cho LLM: đóng gói model base + params + system prompt thành 1 "model riêng" để share team.

Xem file [`Modelfile.anninh`](Modelfile.anninh) (sửa dòng `FROM` thành `qwen3:1.7b` nếu chỉ pull 1.7b), sau đó chạy trong PowerShell:

```powershell
ollama create troly -f 1_ollama_basics/Modelfile.anninh
ollama run troly
```

Hoặc gọi từ Python:

In [ ]:
# Sau khi đã ollama create troly -f Modelfile.anninh, gọi như bình thường
# (Nếu chưa create, cell này sẽ báo lỗi - đó là dấu hiệu cần chạy lệnh trên trước)
try:
    r = ollama.chat(
        model="troly",
        messages=[{"role": "user", "content": "Giải thích điện toán đám mây trong 2 câu."}],
        think=False,
    )
    print(r.message.content)
except ollama.ResponseError:
    print("Chưa tạo model 'troly'. Chạy trong PowerShell:")
    print("  ollama create troly -f 1_ollama_basics/Modelfile.anninh")

## Bài tập (~22 phút thực hành)

1. **Hỏi chủ đề của bạn**: ở Bước 1, đổi câu hỏi sang chủ đề bạn quan tâm (nấu ăn, tóm tắt email, giải thích khái niệm).
2. **Đổi persona**: đổi system prompt thành "trợ lý của riêng bạn" (gia sư, trợ lý code, trợ lý email).
3. **Pull thêm model**: `ollama pull gemma3:4b`, thêm vào `MODELS_TO_TEST` ở Bước 4, so tốc độ + chất lượng.
4. **Build model riêng**: sửa `Modelfile.anninh` (đổi `FROM` về `qwen3:1.7b`, sửa system prompt cho trả lời cực ngắn), `ollama create troly`, test.
5. **(Thêm)** chỉnh `temperature` thấp ↔ cao trong cell Bước 1, quan sát khác biệt.

---

## Tiếp theo
[Module 2 — RAG cho tài liệu của bạn →](../2_rag/notebook.ipynb)